# Problem Book P1, Chapter 12: Asymptotic Adventures (Python companion)

Part A checks the numbers of the chapter, problem by problem. Part B is a laboratory: four experiments that push the
problems further, with exercises first and solutions at the end. We use `numpy`, `matplotlib` and `mpmath` (for high precision,
special functions and numerical integrals).

*Before running a cell, predict its output.*

In [ ]:
import math, random, itertools
from math import comb, factorial, log, sqrt, pi, exp
from fractions import Fraction as F
import numpy as np
import matplotlib.pyplot as plt
import mpmath as mp
mp.mp.dps = 30
random.seed(12); rng = np.random.default_rng(12)

def Q(n):                      # Ramanujan's Q(n) = sum_{k=1}^n n^(k falling)/n^k, in high precision
    s, p = mp.mpf(0), mp.mpf(1)
    for k in range(1, n + 1):
        p *= mp.mpf(n - k + 1) / n; s += p
        if p < mp.mpf(10) ** -40: break
    return s
def R(n):                      # R(n) = 1 + n/(n+1) + n^2/((n+1)(n+2)) + ...
    s, p, k = mp.mpf(0), mp.mpf(1), 0
    while p > mp.mpf(10) ** -40:
        s += p; k += 1; p *= mp.mpf(n) / (n + k)
    return s

## Part A: checks

**Problem 12.1 ($\sum k^n$).** The normalised sum against $\frac e{e-1}-\frac{e(e+1)}{2(e-1)^3n}$; the last column should approach a constant (the next coefficient).

In [ ]:
e = mp.e
for n in [10, 100, 1000]:
    s = sum(mp.mpf(k) ** n for k in range(1, n + 1)) / mp.mpf(n) ** n
    approx = e / (e - 1) - e * (e + 1) / (2 * (e - 1) ** 3 * n)
    print(n, mp.nstr(s, 8), mp.nstr(approx, 8), mp.nstr((s - approx) * n * n, 5))

**Problem 12.2 (involutions).** The saddle-point formula (c) and the closed form (d), as ratios to the exact $t_n$.

In [ ]:
t = [1, 1]
for n in range(1, 3000): t.append(t[n] + n * t[n - 1])
print(t[:10])
for n in [10, 100, 1000, 2000]:
    r = (mp.sqrt(4 * n + 1) - 1) / 2; b = r + 2 * r * r
    saddle = mp.factorial(n) * mp.exp(r + r * r / 2) / (r ** n * mp.sqrt(2 * mp.pi * b))
    closed = mp.mpf(n) ** (mp.mpf(n) / 2) * mp.exp(-mp.mpf(n) / 2 + mp.sqrt(n) - mp.mpf(1) / 4) / mp.sqrt(2)
    print(n, 'saddle ratio', mp.nstr(t[n] / saddle, 6), ' closed ratio', mp.nstr(t[n] / closed, 6), ' (closed-1)*sqrt(n)', mp.nstr((t[n] / closed - 1) * mp.sqrt(n), 4))

**Problem 12.3 (Bell numbers).** $\ln b_n$ against $n(u-1+\frac1u)$ with $ue^u=n$, the bound of (a), and Moser--Wyman's correction $-1-\frac12\ln(1+u)$.

In [ ]:
row, B = [1], [1]
for i in range(1000):
    new = [row[-1]]
    for x in row: new.append(new[-1] + x)
    row = new; B.append(row[0])
print(B[:10])
for n in [10, 100, 500, 1000]:
    u = mp.lambertw(n).real; main = n * (u - 1 + 1 / u)
    lb = mp.log(B[n])
    assert lb <= main - 1                     # part (a)
    L = mp.log(n); LL = mp.log(L)
    print(n, mp.nstr(lb, 10), ' gap', mp.nstr(lb - main, 6), ' MW', mp.nstr(-1 - mp.log(1 + u) / 2, 6), ' expansion (c)', mp.nstr(n * (L - LL - 1 + (1 + LL) / L), 8))

**Problem 12.4 (distinct cycle lengths).** Exact $d_n$ from the product, against $e^{-\gamma}$; and a brute-force check for $n\le7$.

In [ ]:
N = 300
c = [F(0)] * (N + 1); c[0] = F(1)
for k in range(1, N + 1):
    for m in range(N, k - 1, -1):
        c[m] += c[m - k] / k
print([str(c[n]) for n in range(1, 6)])
def cycle_type(p):
    seen, out = set(), []
    for i in range(len(p)):
        if i not in seen:
            j, L = i, 0
            while j not in seen: seen.add(j); j = p[j]; L += 1
            out.append(L)
    return out
for n in range(1, 8):
    cnt = sum(1 for p in itertools.permutations(range(n)) if len(set(ct := cycle_type(p))) == len(ct))
    assert F(cnt, factorial(n)) == c[n]
for n in [10, 50, 100, 300]: print(n, float(c[n]), float(c[n]) - float(mp.exp(-mp.euler)))
print('e^-gamma =', mp.exp(-mp.euler))

**Problem 12.5 (alternating permutations).** The recurrence, a brute-force count, and the relative error of $2(\frac2\pi)^{n+1}$ against $(-1)^{n+1}3^{-(n+1)}$.

In [ ]:
A = [1, 1]
for n in range(1, 40): A.append(sum(comb(n, k) * A[k] * A[n - k] for k in range(n + 1)) // 2)
print(A[:12])
for n in range(1, 9):
    cnt = sum(1 for p in itertools.permutations(range(n)) if all((p[i] < p[i + 1]) == (i % 2 == 0) for i in range(n - 1)))
    assert cnt == A[n]
for n in [5, 10, 20, 30]:
    exact = mp.mpf(A[n]) / mp.factorial(n); approx = 2 * (2 / mp.pi) ** (n + 1)
    print(n, mp.nstr(exact, 10), mp.nstr(exact / approx - 1, 5), mp.nstr(mp.mpf(-1) ** (n + 1) * mp.mpf(3) ** (-(n + 1)), 5))

**Problem 12.6 (powers of binomial coefficients).** Ratios of $\sum_k\binom nk^s$ to the asymptotic formula.

In [ ]:
for s in [2, 3, 4]:
    out = []
    for n in [10, 100, 1000]:
        S = sum(mp.mpf(comb(n, k)) ** s for k in range(n + 1))
        out.append(mp.nstr(S / (mp.mpf(2) ** (s * n) * (2 / (mp.pi * n)) ** (mp.mpf(s - 1) / 2) / mp.sqrt(s)), 6))
    print(s, out)

**Problem 12.7 (a triple birthday).** The integral formula against exact enumeration for small $n$, the values for $n=365$, and the ratio to $\Gamma(\frac43)(6n^2)^{1/3}$.

In [ ]:
def ET(n, m):
    f = lambda t: mp.exp(-t) * sum((t / n) ** j / mp.factorial(j) for j in range(m)) ** n
    return mp.quad(f, [0, 10 * n ** (1 - 1 / mp.mpf(m)), mp.inf])
def exact(n, m):
    tot, k = F(0), 0
    while True:
        cnt = sum(1 for s in itertools.product(range(n), repeat=k) if all(s.count(v) < m for v in range(n)))
        if cnt == 0: return tot
        tot += F(cnt, n ** k); k += 1
for n, m in [(3, 2), (3, 3), (4, 3)]:
    print(n, m, float(exact(n, m)), mp.nstr(ET(n, m), 12))
print('n=365: pair', mp.nstr(ET(365, 2), 8), '= 1+Q(365) =', mp.nstr(1 + Q(365), 8), '; triple', mp.nstr(ET(365, 3), 8), '; approx', mp.nstr(mp.gamma(mp.mpf(4) / 3) * (6 * 365 ** 2) ** (mp.mpf(1) / 3), 6))
for n in [10 ** 3, 10 ** 4, 10 ** 5]:
    print(n, mp.nstr(ET(n, 3) / (mp.gamma(mp.mpf(4) / 3) * (6 * mp.mpf(n) ** 2) ** (mp.mpf(1) / 3)), 6))

**Problem 12.8 (Ramanujan's $Q$, $R$ and $\theta_n$).** The identities of (a) and (b), the expansions of (d), and the sum of (e).

In [ ]:
for n in [1, 5, 10, 100, 1000, 10000]:
    q, r = Q(n), R(n)
    assert abs(q + r - mp.factorial(n) * mp.exp(n) / mp.mpf(n) ** n) < mp.mpf(10) ** -20 * (q + r)
    print(n, 'theta', mp.nstr((r - q) / 2, 10), ' Q-sqrt(pi n/2)', mp.nstr(q - mp.sqrt(mp.pi * n / 2), 8), ' R-sqrt(pi n/2)', mp.nstr(r - mp.sqrt(mp.pi * n / 2), 8))
n = 7
print(mp.nstr(Q(n), 15), mp.nstr(n * mp.quad(lambda s: s ** (n - 1) * mp.exp(n * (1 - s)), [1, mp.inf]), 15))
print(mp.nstr(R(n), 15), mp.nstr(n * mp.quad(lambda s: s ** (n - 1) * mp.exp(n * (1 - s)), [0, 1]), 15))
for n in [100, 10000]:
    s = mp.exp(-n) * mp.nsum(lambda k: mp.mpf(n) ** k / mp.factorial(k), [0, n])
    print(n, mp.nstr(s, 12), mp.nstr(mp.mpf(1) / 2 + mp.mpf(2) / (3 * mp.sqrt(2 * mp.pi * n)), 12))

**Problem 12.9 (coupons $m$ times).** The integral against $nH_n$ for $m=1$, and the gap $\frac1n\bigl(\mathbb E[T_m]-n\ln n-(m-1)n\ln\ln n\bigr)$ against Erd\H os--R\'enyi's constant $\gamma-\ln(m-1)!$.

In [ ]:
def Edixie(n, m):
    em = lambda x: sum(x ** j / mp.factorial(j) for j in range(m))
    f = lambda x: 1 - (1 - mp.exp(-x) * em(x)) ** n
    L = mp.log(n)
    return n * mp.quad(f, [0, L / 2, L, L + (m - 1) * mp.log(L + 1), L + (m + 5) * mp.log(L + 2) + 10, mp.inf])
print(mp.nstr(Edixie(10, 1), 12), mp.nstr(10 * mp.harmonic(10), 12))
for m in [2, 3]:
    for n in [10 ** 2, 10 ** 4, 10 ** 6]:
        E = Edixie(n, m)
        print(m, n, mp.nstr((E - n * mp.log(n) - (m - 1) * n * mp.log(mp.log(n))) / n, 6), ' constant', mp.nstr(mp.euler - mp.log(mp.factorial(m - 1)), 6))

**Problem 12.10 (random maps).** Brute force over all maps for $n\le5$, then $C_n$ against $\frac12(\ln2n+\gamma)$.

In [ ]:
def brute(n):
    cyc = comp = 0
    for f in itertools.product(range(n), repeat=n):
        cp = set()
        for x in range(n):
            y = x
            for _ in range(n): y = f[y]      # y is now cyclic
            z = y
            while True:
                cp.add(z); z = f[z]
                if z == y: break
        seen, k = set(), 0
        for x in cp:
            if x not in seen:
                k += 1; y = x
                while y not in seen: seen.add(y); y = f[y]
        cyc += len(cp); comp += k
    return F(cyc, n ** n), F(comp, n ** n)
def formulas(n):
    q = cc = F(0); p = F(1)
    for k in range(1, n + 1):
        p *= F(n - k + 1, n); q += p; cc += p / k
    return q, cc
for n in range(1, 6): assert brute(n) == formulas(n)
for n in [10, 100, 10 ** 4, 10 ** 6]:
    cc, p = mp.mpf(0), mp.mpf(1)
    for k in range(1, n + 1):
        p *= mp.mpf(n - k + 1) / n; cc += p / k
        if p < 1e-40: break
    approx = (mp.log(2 * n) + mp.euler) / 2
    print(n, mp.nstr(cc, 8), mp.nstr(approx, 8), ' diff*sqrt(n)', mp.nstr((cc - approx) * mp.sqrt(n), 5))

**Problem 12.11 (parking).** Parking functions counted by brute force, the law of $K$ for $n=5$, the Borel limit and its tail, and $\mathbb E[K]/\sqrt n$.

In [ ]:
def is_pf(a): return all(x <= i + 1 for i, x in enumerate(sorted(a)))
for n in range(1, 7):
    assert sum(1 for a in itertools.product(range(1, n + 1), repeat=n) if is_pf(a)) == (n + 1) ** (n - 1)
def first_empty(a, n):
    occ = [False] * (n + 3)
    for p in a:
        while p <= n + 1 and occ[p]: p += 1
        if p <= n + 1: occ[p] = True
    j = 1
    while occ[j]: j += 1
    return j - 1
n = 5
cnt = [0] * (n + 1)
for a in itertools.product(range(1, n + 2), repeat=n): cnt[first_empty(a, n)] += 1
print(cnt, [comb(n, k) * (k + 1) ** max(k - 1, 0) * (n - k) ** (n - k) for k in range(n + 1)])
p = [mp.mpf(k + 1) ** (k - 1) * mp.exp(-(k + 1)) / mp.factorial(k) for k in range(2000)]
print('sum of p_k up to 2000:', mp.nstr(sum(p), 8), ' tail estimate sqrt(2/(pi*2000)):', mp.nstr(mp.sqrt(2 / (mp.pi * 2000)), 4))
print('p_k * sqrt(2 pi) k^1.5:', [mp.nstr(p[k] * mp.sqrt(2 * mp.pi) * k ** 1.5, 5) for k in (10, 100, 1000)])
for n in [10, 100, 1000, 3000]:
    EK = sum(F(comb(n, k) * (k + 1) ** (k - 1) * (n - k) ** (n - k) * k, (n + 1) ** n) for k in range(1, n + 1))
    print(n, float(EK), float(EK) / sqrt(n))

**Problem 12.12 ($x+\frac1x$).** $x_n^2-2n-\frac12\ln n$ settles at $C$.

In [ ]:
x = 1.0; marks = {10, 100, 1000, 10 ** 4, 10 ** 5, 10 ** 6}
for n in range(1, 10 ** 6 + 1):
    x += 1 / x
    if n in marks: print(n, x * x - 2 * n - log(n) / 2)

**Problem 12.13 (sine iteration).** $\frac1{u_n^2}-\frac n3$ grows like $\frac15\ln n$: its increments over factors of $10$ approach $\frac15\ln10=0.4605$.

In [ ]:
u = mp.mpf(1); prev = None; marks = {10 ** 3, 10 ** 4, 10 ** 5, 10 ** 6}
mp.mp.dps = 20
for n in range(1, 10 ** 6 + 1):
    u = mp.sin(u)
    if n in marks:
        v = 1 / u ** 2 - mp.mpf(n) / 3
        print(n, mp.nstr(v, 8), '' if prev is None else mp.nstr(v - prev, 6)); prev = v
mp.mp.dps = 30
print('0.2 ln 10 =', 0.2 * log(10))

**Problem 12.14 ($\tan x=x$).** The roots against the expansion; the scaled error approaches $-\frac{13}{15}$; and $\sum\frac1{x_k^2}$.

In [ ]:
for k in [1, 2, 5, 10, 100]:
    a = (k + mp.mpf(1) / 2) * mp.pi
    r = mp.findroot(lambda x: mp.sin(x) - x * mp.cos(x), a - 1 / a)
    approx = a - 1 / a - 2 / (3 * a ** 3)
    print(k, mp.nstr(r, 10), mp.nstr(approx, 10), mp.nstr((r - approx) * a ** 5, 6))
S = 0
for k in range(1, 2001):
    a = (k + 0.5) * math.pi; S += 1 / (a - 1 / a - 2 / (3 * a ** 3)) ** 2
print('sum of 1/x_k^2 up to 2000:', S, ' (tail about 1/(2000 pi^2) =', 1 / (2000 * math.pi ** 2), ')')

**Problem 12.15 ($n$-step Fibonacci).** The roots $r_n$ against $2-2^{-n}-n2^{-2n-1}$, and the growth of sequences without $n$ heads in a row.

In [ ]:
for n in [2, 3, 5, 10, 20]:
    r = mp.findroot(lambda x: x ** (n + 1) - 2 * x ** n + 1, 2 - mp.mpf(2) ** -n)
    print(n, mp.nstr(r, 15), mp.nstr(2 - mp.mpf(2) ** -n - n * mp.mpf(2) ** (-2 * n - 1), 15))
n = 3; a = [2 ** j for j in range(n)]
for l in range(n, 60): a.append(sum(a[-n:]))
brute = [sum(1 for s in itertools.product('HT', repeat=l) if 'HHH' not in ''.join(s)) for l in range(12)]
assert brute == a[:12]
print([a[l + 1] / a[l] for l in (20, 40, 58)])

**Problem 12.16 ($\sum\sqrt k$).** The remainders after the expansion are of the size of the next term.

In [ ]:
z1, z2 = mp.zeta(-0.5), mp.zeta(0.5); print(z1, z2)
for n in [10, 100, 1000]:
    S1 = mp.nsum(lambda k: mp.sqrt(k), [1, n]); S2 = mp.nsum(lambda k: 1 / mp.sqrt(k), [1, n])
    r1 = S1 - (mp.mpf(2) / 3 * n ** 1.5 + mp.sqrt(n) / 2 + z1 + 1 / (24 * mp.sqrt(n)))
    r2 = S2 - (2 * mp.sqrt(n) + z2 + 1 / (2 * mp.sqrt(n)) - 1 / (24 * mp.mpf(n) ** 1.5))
    print(n, mp.nstr(r1 * mp.mpf(n) ** 2.5, 6), mp.nstr(r2 * mp.mpf(n) ** 3.5, 6))
print('-zeta(3/2)/(4 pi) =', -mp.zeta(1.5) / (4 * mp.pi))

**Problem 12.17 (hyperfactorial).** Glaisher's constant from $\zeta'(-1)$, and the error after the $\frac1{720n^2}$ term.

In [ ]:
A = mp.exp(mp.mpf(1) / 12 - mp.zeta(-1, derivative=1)); print('A =', A, ' mpmath glaisher =', mp.glaisher)
print('ln H(10) =', sum(k * log(k) for k in range(1, 11)))
for n in [10, 100, 1000]:
    lh = mp.fsum(k * mp.log(k) for k in range(1, n + 1))
    base = (mp.mpf(n) ** 2 / 2 + mp.mpf(n) / 2 + mp.mpf(1) / 12) * mp.log(n) - mp.mpf(n) ** 2 / 4 + mp.log(A)
    print(n, mp.nstr(lh - base, 8), mp.nstr(1 / (720 * mp.mpf(n) ** 2), 8), mp.nstr((lh - base - 1 / (720 * mp.mpf(n) ** 2)) * mp.mpf(n) ** 4, 5))
print("zeta'(2) check:", mp.zeta(2, derivative=1), mp.pi ** 2 / 6 * (mp.euler + mp.log(2 * mp.pi) - 12 * mp.log(A)))

**Problem 12.18 (die totals).** Exact $u_n$, the roots of $6-z-\dots-z^6$, and a simulation.

In [ ]:
u = [F(1)]
for n in range(1, 61): u.append(sum(u[n - j] for j in range(1, 7) if n - j >= 0) / 6)
print([str(x) for x in u[1:7]], float(u[7]))
roots = np.roots([-1, -1, -1, -1, -1, -1, 6])     # -z^6 - ... - z + 6, highest degree first
print(sorted(abs(roots)))
print('rho =', 1 / sorted(abs(roots))[1], ' u_20 - 2/7 =', float(u[20] - F(2, 7)))
hits = np.zeros(31); T = 20000
for _ in range(T):
    s = 0
    while s < 30:
        s += random.randint(1, 6)
        if s <= 30: hits[s] += 1
print('simulated u_30 =', hits[30] / T, ' exact', float(u[30]))

**Problem 12.19 (Banach's matchboxes).** The law of $R$ sums to $1$, its mean has the closed form, and the asymptotic formula; plus a simulation for $N=50$.

In [ ]:
for N in [1, 2, 3, 10, 50, 1000]:
    a = [F(comb(2 * N - r, N), 2 ** (2 * N - r)) for r in range(N + 1)]
    assert sum(a) == 1
    E = sum(r * x for r, x in enumerate(a)); assert E == F((2 * N + 1) * comb(2 * N, N), 4 ** N) - 1
    print(N, float(E), 2 * sqrt(N / pi) - 1)
def matchbox(N):
    box = [N, N]
    while True:
        i = random.randint(0, 1)
        if box[i] == 0: return box[1 - i]
        box[i] -= 1
print('simulated, N=50:', np.mean([matchbox(50) for _ in range(20000)]))

**Problem 12.20 (dimers).** The recurrence against the closed form, and the distance to $(n+2)e^{-2}$.

In [ ]:
V = [F(0), F(1)]
for n in range(2, 40): V.append(F(2, n - 1) * sum(V[:n - 1]))
for n in range(10): assert V[n] == sum(F((n - k) * (-2) ** k, factorial(k)) for k in range(n))
for n in [5, 10, 20, 30]:
    d = mp.mpf(V[n].numerator) / V[n].denominator - (n + 2) * mp.exp(-2)
    print(n, mp.nstr(d, 5), mp.nstr(mp.mpf(2) ** (n + 1) / mp.factorial(n + 1), 5))

**Problem 12.21 (self-avoiding walks).** Counts up to $n=14$, the two lower-bound families, and the bounds.

In [ ]:
def saw(N):
    cnt = [0] * (N + 1)
    def go(x, y, vis, k):
        cnt[k] += 1
        if k == N: return
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            p = (x + dx, y + dy)
            if p not in vis:
                vis.add(p); go(p[0], p[1], vis, k + 1); vis.remove(p)
    go(0, 0, {(0, 0)}, 0)
    return cnt
c = saw(14); print(c)
assert all(c[m + n] <= c[m] * c[n] for m in range(1, 8) for n in range(1, 7))
e_, v_ = 1, 2; b = [3]
for n in range(2, 15): e_, v_ = e_ + v_, 2 * e_ + v_; b.append(e_ + v_)
print('E/N/S walks:', b[:6], ' ratio', b[-1] / b[-2], ' 1+sqrt2 =', 1 + sqrt(2))
print('upper bounds c_n^(1/n):', [round(c[n] ** (1 / n), 4) for n in (10, 12, 14)])

**Problem 12.22 (common subsequences).** Exact $\mathbb E[L_n]$ for $n\le8$ by enumeration, and the root $\kappa^*$.

In [ ]:
def lcs(a, b):
    m, n = len(a), len(b); D = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            D[i + 1][j + 1] = D[i][j] + 1 if a[i] == b[j] else max(D[i][j + 1], D[i + 1][j])
    return D[m][n]
for n in range(1, 9):
    seqs = list(itertools.product((0, 1), repeat=n))
    tot = sum(lcs(a, b) for a in seqs for b in seqs)
    print(n, F(tot, 4 ** n), round(tot / 4 ** n / n, 4))
H = lambda k: -k * mp.log(k) - (1 - k) * mp.log(1 - k)
print('kappa* =', mp.findroot(lambda k: 2 * H(k) - k * mp.log(2), 0.9))

## Part B: laboratory

**L12.1 (Ramanujan's next term).** Compute $\theta_n$ for $n$ up to $5000$ and plot $n(\theta_n-\frac13)$. Guess its limit as a simple fraction. Where in the
proof of Problem 12.8 would the next term come from?

**L12.2 (birthdays and coupons by simulation).** Simulate $20000$ rooms filling up until a triple birthday ($n=365$) and compare with $88.74$. Then
simulate the double dixie cup for $n=50$ types and compare with the integral of Problem 12.9 and with the asymptotic formula including
the Erd\H os--R\'enyi constant. Which is closer at $n=50$?

**L12.3 (parking, discrete and continuous).** Simulate random dimers on a row of $10^5$ sites (use the labels of Problem 12.20) and compare the fraction
of empty sites with $e^{-2}$. Then simulate R\'enyi's continuous parking: cars of length $1$ arrive at uniform positions on a street of length $x$ and
park if there is room. Estimate the filled fraction for $x=1000$, and compare with $\int_0^\infty\exp\bigl(-2\int_0^t\frac{1-e^{-u}}u\dd u\bigr)\dd t$.

**L12.4 (the Chv\'atal--Sankoff constant).** Estimate $\frac{\mathbb E[L_n]}n$ for random binary words of lengths $n=100,\dots,4000$, using the bit-parallel algorithm
below, which updates a bit vector $V$ with $U=V\wedge M_b$, $V\leftarrow(V+U)\vee(V-U)$ for each letter $b$ of the second word, where $M_b$ marks the positions of $b$
in the first word; the LCS is the number of zero bits of $V$ among the $n$ lowest. Check it against the dynamic programme first. Where does the
estimate sit relative to Lueker's interval $[0.788,0.826]$?

## Solutions

### L12.1

In [ ]:
mp.mp.dps = 50
ns = list(range(10, 5001, 10)); vals = []
for n in ns:
    th = (mp.factorial(n) * mp.exp(n) / mp.mpf(n) ** n - 2 * Q(n)) / 2
    vals.append(float(n * (th - mp.mpf(1) / 3)))
mp.mp.dps = 30
plt.plot(ns, vals); plt.axhline(4 / 135, ls='--', color='k'); plt.xlabel('n'); plt.ylabel('n(theta_n - 1/3)'); plt.show()
print(vals[-1], 4 / 135)

The limit is $\frac4{135}=0.0296296$. In Problem 12.8(c), push the expansion one step further: $g(v)=1-\frac v3+\frac{v^2}{12}-\frac{2v^3}{135}+\cdots$ (the notebook can
check it by series inversion). The even powers contribute equally to $Q$ and $R$; the odd ones with opposite signs. So $R-Q=\frac23+2\cdot\frac2{135}\cdot n\int_0^\infty v^3e^{-nv^2/2}\dd v+\cdots=\frac23+\frac8{135n}+\cdots$,
and $\theta_n=\frac13+\frac4{135n}+\cdots$

In [ ]:
# check the series of g(v) = v/(s(v)-1), where s - 1 - ln s = v^2/2, numerically at v = +-0.01
def s_of_v(v):
    return mp.findroot(lambda s: s - 1 - mp.log(s) - v * v / 2, 1 + v + v * v / 3)
for v in [mp.mpf('0.01'), mp.mpf('-0.01')]:
    g = v / (s_of_v(v) - 1)
    print(mp.nstr(g, 15), mp.nstr(1 - v / 3 + v ** 2 / 12 - 2 * v ** 3 / 135, 15))

### L12.2

In [ ]:
def triple(n):
    cnt = np.zeros(n, int); k = 0
    while True:
        i = rng.integers(n); cnt[i] += 1; k += 1
        if cnt[i] == 3: return k
sims = [triple(365) for _ in range(20000)]
print('triple birthday:', np.mean(sims), '+-', 2 * np.std(sims) / sqrt(len(sims)))
def dixie(n, m):
    cnt = np.zeros(n, int); k = 0; done = 0
    while done < n:
        i = rng.integers(n); cnt[i] += 1; k += 1
        if cnt[i] == m: done += 1
    return k
sims = [dixie(50, 2) for _ in range(5000)]
n = 50
print('double dixie n=50:', np.mean(sims), ' integral', mp.nstr(Edixie(50, 2), 8), ' asymptotic', n * log(n) + n * log(log(n)) + n * float(mp.euler))

The simulation agrees with the integral within its error bar. At $n=50$ the three-term asymptotic formula is still about $0.65n$ off, because the
$o(n)$ term decays like $\frac{\ln\ln n}{\ln n}$: asymptotic formulas with $\ln\ln n$ in them are honest but slow.

### L12.3

In [ ]:
def dimers(n):
    labels = rng.random(n - 1); order = np.argsort(labels)
    occ = np.zeros(n, bool)
    for i in order:
        if not occ[i] and not occ[i + 1]: occ[i] = occ[i + 1] = True
    return 1 - occ.mean()
print('dimers, n=1e5:', np.mean([dimers(10 ** 5) for _ in range(5)]), ' e^-2 =', exp(-2))
import sys; sys.setrecursionlimit(10000)
def renyi(x):          # cars of length 1 on [0, x], filled recursively: the first car splits the street
    stack, filled = [x], 0
    while stack:
        L = stack.pop()
        if L < 1: continue
        a = random.uniform(0, L - 1); filled += 1
        stack.append(a); stack.append(L - 1 - a)
    return filled / x
print('Renyi, x=1000:', np.mean([renyi(1000) for _ in range(200)]))
inner = lambda t: mp.quad(lambda u: (1 - mp.exp(-u)) / u, [0, t])
print('Renyi constant:', mp.quad(lambda t: mp.exp(-2 * inner(t)), [0, 1, 10, mp.inf]))

The continuous street fills less ($74.8\%$) than the discrete row ($86.5\%$): in the continuum a car can leave a gap of any length below $1$, wasted for ever.
Splitting at the first car makes the simulation exact, for the same reason as in Problem 12.20: given the first car, the two sides evolve
independently.

### L12.4

In [ ]:
def lcs_bits(a, b):
    n = len(a); mask = (1 << n) - 1
    M = {0: 0, 1: 0}
    for i, ch in enumerate(a): M[ch] |= 1 << i
    V = mask
    for ch in b:
        U = V & M[ch]
        V = ((V + U) | (V - U)) & mask
    return n - bin(V).count('1')
for _ in range(200):
    a = [random.randint(0, 1) for _ in range(30)]; b = [random.randint(0, 1) for _ in range(30)]
    assert lcs_bits(a, b) == lcs(a, b)
for n in [100, 500, 1000, 2000, 4000]:
    reps = 40 if n <= 1000 else 10
    est = np.mean([lcs_bits([random.randint(0, 1) for _ in range(n)], [random.randint(0, 1) for _ in range(n)]) for _ in range(reps)]) / n
    print(n, round(est, 4))

The estimates climb slowly (superadditivity says that $\frac{\mathbb E[L_n]}n$ is always below the limit) and reach about $0.81$, inside Lueker's interval. The
correction is of order $\frac1{\sqrt n}$ or smaller, which is why the exact value is so hard to pin down. Bit-parallel algorithms of this kind go back to Allison and Dix
(1986); they process $n$ positions per machine operation: here Python's big integers play the role of very long machine words.